# 02 — MedNeXt
**Modelo:** MedNeXt v2 (integradp en nnUn-Net v2)  
**Dataset:** VerSe 2019+2020 — 355 pacientes, 374 CTs  
**Configuraciones:** 2D, 3D lowres, 3D fullres  
**Epochs:** 250 máximo + early stopping (patience=50)  
**Folds:** 5-fold cross-validation  
**Métricas:** DSC + ID Rate + RMSD  

### Flujo
1. Setup + restaurar preprocessing desde Drive
2. Entrenar 2D (5 folds)
3. Entrenar 3D lowres (5 folds)
4. Entrenar 3D fullres (5 folds)
5. Inferencia 2D + 3D lowres + 3D fullres
6. Evaluación completa

# Montar Drive e instalar dependencias

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [ ]:
!nvidia-smi

Fri Apr 24 09:51:01 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   35C    P8             12W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# MedNeXt v2 ya viene con nnU-Net v2 desde diciembre 2025
!pip install nnunetv2 nibabel scipy pandas tqdm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.6/205.6 kB 7.9 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.0/77.0 kB 10.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.1/63.1 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.7/73.7 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.6/52.6 MB 50.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.5/26.5 MB 98.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 110.8 MB/s eta 0:00:0

In [ ]:
# Verificar que MedNeXt está disponible
import subprocess
result = subprocess.run(
    ['python', '-c', 'from nnunetv2.training.nnUNetTrainer.variants.network_architecture.nnUNetTrainerMedNeXt import nnUNetTrainerMedNeXt; print("✓ MedNeXt disponible")'],
    capture_output=True, text=True
)
print(result.stdout if result.returncode == 0 else '⚠ MedNeXt no encontrado — se instalará manualmente')

⚠ MedNeXt no encontrado — se instalará manualmente


# Configurar rutas

In [ ]:
import os
from pathlib import Path

DRIVE_BASE    = '/content/drive/MyDrive/VerSe_2020_Dataset/preprocessed_verse_for_training'
DRIVE_RESULTS = '/content/drive/MyDrive/VerSe_2020_Dataset/results'

BASE_DIR       = '/content/nnunet_verse'
NNUNET_RAW     = f'{BASE_DIR}/nnUNet_raw'
NNUNET_PREPROC = f'{BASE_DIR}/nnUNet_preprocessed'
NNUNET_RESULTS = f'{BASE_DIR}/nnUNet_results'

for d in [NNUNET_RAW, NNUNET_PREPROC, NNUNET_RESULTS, DRIVE_RESULTS]:
    os.makedirs(d, exist_ok=True)

os.environ['nnUNet_raw']          = NNUNET_RAW
os.environ['nnUNet_preprocessed'] = NNUNET_PREPROC
os.environ['nnUNet_results']      = NNUNET_RESULTS
os.environ['nnUNet_n_proc_DA'] = '16'

DATASET_ID   = 507
DATASET_NAME = f'Dataset{DATASET_ID:03d}_VerSe2020'
TRAINER = 'nnUNetTrainerMedNeXt_250epochs'

print('✓ Rutas configuradas')
print(f'  Drive preprocessing: {DRIVE_BASE}')
print(f'  Drive results:       {DRIVE_RESULTS}')
print(f'  nnUNet_results:      {NNUNET_RESULTS}')

✓ Rutas configuradas
  Drive preprocessing: /content/drive/MyDrive/VerSe_2020_Dataset/preprocessed_verse_for_training
  Drive results:       /content/drive/MyDrive/VerSe_2020_Dataset/results
  nnUNet_results:      /content/nnunet_verse/nnUNet_results


# Restaurar preprocessing desde Drive

In [ ]:
import subprocess
import os
from pathlib import Path
from tqdm.notebook import tqdm
import threading

def copy_with_rsync_fast(src, dst, desc='Copiando'):
    src = Path(src)
    dst = Path(dst)
    dst.mkdir(parents=True, exist_ok=True)

    # Contar archivos y tamaño total
    all_files = [f for f in src.rglob('*') if f.is_file()]
    total_files = len(all_files)
    total_size = sum(f.stat().st_size for f in all_files)
    total_gb = round(total_size / 1e9, 2)
    print(f'{desc}: {total_files} archivos ({total_gb} GB)')

    # Barra de progreso en hilo separado
    copied_files = [0]
    stop_flag = [False]

    def update_progress(pbar):
        while not stop_flag[0]:
            current = len(list(dst.rglob('*')))
            pbar.n = min(current, total_files)
            pbar.refresh()
            threading.Event().wait(1.0)  # actualiza cada segundo

    with tqdm(total=total_files, unit='archivos', desc=desc,
              bar_format='{l_bar}{bar}| {n_fmt}/{total_fmt} [{elapsed}<{remaining}, {rate_fmt}]') as pbar:

        t = threading.Thread(target=update_progress, args=(pbar,))
        t.start()

        cmd = [
            'rsync', '-r',
            '--info=progress2',
            '--no-perms',
            '--no-owner',
            '--no-group',
            '--inplace',
            '--whole-file',
            str(src) + '/',
            str(dst) + '/'
        ]
        result = subprocess.run(cmd, capture_output=True, text=True)

        stop_flag[0] = True
        t.join()
        pbar.n = total_files
        pbar.refresh()

    if result.returncode == 0:
        print(f'✓ {desc} completado')
    else:
        print(f'✗ Error en {desc}')
        print(result.stderr)

# Instalar rsync
!apt-get install -y rsync -q

print('Restaurando preprocessing desde Drive...\n')

src_pre = Path(DRIVE_BASE) / 'nnUNet_preprocessed' / DATASET_NAME
dst_pre = Path(NNUNET_PREPROC) / DATASET_NAME
if src_pre.exists():
    copy_with_rsync_fast(src_pre, dst_pre, 'nnUNet_preprocessed')
    print('✓ Preprocessing restaurado\n')
else:
    print(f'✗ No encontrado: {src_pre}')

src_raw = Path(DRIVE_BASE) / 'nnUNet_raw' / DATASET_NAME
dst_raw = Path(NNUNET_RAW) / DATASET_NAME
if src_raw.exists():
    copy_with_rsync_fast(src_raw, dst_raw, 'nnUNet_raw')
    print('✓ nnUNet_raw restaurado\n')
else:
    print(f'✗ No encontrado: {src_raw}')

n_pre = len(list((dst_pre / 'nnUNetPlans_3d_lowres').glob('*')))
n_img = len(list((dst_raw / 'imagesTs').glob('*.nii.gz')))
n_lbl = len(list((dst_raw / 'labelsTs').glob('*.nii.gz')))
print(f'✓ Verificación final:')
print(f'  Preprocessing 3D lowres: {n_pre} archivos')
print(f'  imagesTs: {n_img} CTs')
print(f'  labelsTs: {n_lbl} máscaras')

Reading package lists...
Building dependency tree...
Reading state information...
rsync is already the newest version (3.2.7-0ubuntu0.22.04.4).
0 upgraded, 0 newly installed, 0 to remove and 42 not upgraded.
Restaurando preprocessing desde Drive...

nnUNet_preprocessed: 2613 archivos (49.07 GB)


nnUNet_preprocessed:   0%|          | 0/2613 [00:00<?, ?archivos/s]

# Instalar trainer MedNeXt 250 epochs

In [ ]:
import nnunetv2
from pathlib import Path

nnunet_dir = Path(nnunetv2.__file__).parent

# Verificar si MedNeXt ya existe nativamente en nnU-Net v2
mednext_paths = list(nnunet_dir.rglob('*MedNeXt*'))
print(f'Archivos MedNeXt encontrados: {len(mednext_paths)}')
for p in mednext_paths:
    print(f'  {p.relative_to(nnunet_dir)}')

# Instalar trainer custom con 500 epochs
trainer_dir = nnunet_dir / 'training' / 'nnUNetTrainer' / 'variants' / 'network_architecture'
trainer_dir.mkdir(parents=True, exist_ok=True)

trainer_code = '''
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer

try:
    # Intentar usar MedNeXt nativo de nnU-Net v2
    from nnunetv2.training.nnUNetTrainer.variants.network_architecture.nnUNetTrainerMedNeXt import nnUNetTrainerMedNeXt
    BASE_TRAINER = nnUNetTrainerMedNeXt
    print("Usando MedNeXt nativo de nnU-Net v2")
except ImportError:
    # Fallback: usar arquitectura MedNeXt de nnunet_mednext
    from nnunet_mednext import create_mednext_v1
    BASE_TRAINER = nnUNetTrainer
    print("Usando MedNeXt v1 como fallback")

class nnUNetTrainerMedNeXt_250epochs(BASE_TRAINER):
    """
    MedNeXt con 500 epochs + early stopping (patience=50)
    VerSe 2019+2020 — Benchmark vertebral segmentation
    """
    max_num_epochs = 250
'''

trainer_path = trainer_dir / 'nnUNetTrainerMedNeXt_250epochs.py'
trainer_path.write_text(trainer_code)
print(f'\n✓ Trainer instalado: {trainer_path}')

# Verificar e instalar MedNeXt si no está nativo

In [ ]:
import subprocess
import nnunetv2
from pathlib import Path

nnunet_dir  = Path(nnunetv2.__file__).parent
trainer_dir = nnunet_dir / 'training' / 'nnUNetTrainer' / 'variants' / 'network_architecture'
trainer_dir.mkdir(parents=True, exist_ok=True)
trainer_path = trainer_dir / 'nnUNetTrainerMedNeXt_250epochs.py'

# Verificar si MedNeXt nativo está disponible
result = subprocess.run(
    ['python', '-c', 'from nnunetv2.training.nnUNetTrainer.variants.network_architecture.nnUNetTrainerMedNeXt import nnUNetTrainerMedNeXt; print("✓ MedNeXt nativo disponible")'],
    capture_output=True, text=True
)

if result.returncode == 0:
    print(result.stdout)
    trainer_code = '''
from nnunetv2.training.nnUNetTrainer.variants.network_architecture.nnUNetTrainerMedNeXt import nnUNetTrainerMedNeXt

class nnUNetTrainerMedNeXt_250epochs(nnUNetTrainerMedNeXt):
    """
    MedNeXt v2 nativo en nnU-Net v2
    250 epochs por fold — 1250 totales
    Early stopping patience=50
    """
    max_num_epochs = 250
'''
else:
    print('MedNeXt nativo no disponible, instalando nnunet_mednext...')
    !pip install git+https://github.com/MIC-DKFZ/MedNeXt.git -q
    print('✓ nnunet_mednext instalado')

    trainer_code = '''
from nnunetv2.training.nnUNetTrainer.nnUNetTrainer import nnUNetTrainer
from nnunet_mednext import create_mednext_v1

class nnUNetTrainerMedNeXt_250epochs(nnUNetTrainer):
    """
    MedNeXt v1 integrado en nnU-Net v2
    250 epochs por fold — 1250 totales
    Early stopping patience=50
    Kernel size=3, Model size=B
    """
    max_num_epochs = 250

    def build_network_architecture(self, architecture_class_name, arch_init_kwargs,
                                   arch_init_kwargs_req_import, num_input_channels,
                                   num_output_channels, enable_deep_supervision):
        model = create_mednext_v1(
            num_channels=num_input_channels,
            num_classes=num_output_channels,
            model_id='B',
            kernel_size=3,
            deep_supervision=enable_deep_supervision
        )
        return model
'''

trainer_path.write_text(trainer_code)

# Verificar instalación
result = subprocess.run(
    ['python', '-c', 'from nnunetv2.training.nnUNetTrainer.variants.network_architecture.nnUNetTrainerMedNeXt_250epochs import nnUNetTrainerMedNeXt_250epochs; print(f"✓ Trainer instalado: max_epochs={nnUNetTrainerMedNeXt_250epochs.max_num_epochs}")'],
    capture_output=True, text=True
)
print(result.stdout if result.returncode == 0 else f'✗ Error: {result.stderr}')

# Cambiar bacth dependiendo de la GPU

In [ ]:
import json
from pathlib import Path

plans_path = Path(NNUNET_PREPROC) / DATASET_NAME / 'nnUNetPlans.json'

with open(plans_path) as f:
    plans = json.load(f)

print(f'Batch size actual 2D: {plans["configurations"]["2d"]["batch_size"]}')

# L4 24GB — MedNeXt es más pesado que nnU-Net
plans['configurations']['2d']['batch_size'] = 40

with open(plans_path, 'w') as f:
    json.dump(plans, f, indent=2)

print(f'✓ Batch size actualizado a 40')

# Funciones de backup y restauración

In [ ]:
import shutil
import subprocess
from pathlib import Path

MODEL_FOLDER = 'mednext'  # ← único cambio entre notebooks

def backup_fold(config, fold):
    local = Path(NNUNET_RESULTS) / DATASET_NAME / \
            f'{TRAINER}__nnUNetPlans__{config}' / f'fold_{fold}'
    dest  = Path(DRIVE_RESULTS) / MODEL_FOLDER / DATASET_NAME / \
            f'{TRAINER}__nnUNetPlans__{config}' / f'fold_{fold}'

    if not local.exists():
        print(f'  ⚠ No existe local fold_{fold}, saltando backup')
        return

    # No sobreescribir si ya tiene checkpoint_final en Drive
    if (dest / 'checkpoint_final.pth').exists():
        print(f'  ⚠ fold_{fold} ya completado en Drive, saltando backup')
        return

    dest.mkdir(parents=True, exist_ok=True)
    shutil.copytree(str(local), str(dest), dirs_exist_ok=True)
    print(f'  ✓ Backup fold_{fold} ({config}) → Drive')

def restore_config(config):
    src  = Path(DRIVE_RESULTS) / MODEL_FOLDER / DATASET_NAME / \
           f'{TRAINER}__nnUNetPlans__{config}'
    dest = Path(NNUNET_RESULTS) / DATASET_NAME / \
           f'{TRAINER}__nnUNetPlans__{config}'
    if src.exists():
        dest.mkdir(parents=True, exist_ok=True)
        shutil.copytree(str(src), str(dest), dirs_exist_ok=True)
        print(f'✓ Checkpoints restaurados: {config}')
    else:
        print(f'  No hay checkpoints en Drive para {config}')

def train_fold(config, fold):
    # Verificar si el fold ya está completado en Drive
    checkpoint = Path(DRIVE_RESULTS) / MODEL_FOLDER / DATASET_NAME / \
                 f'{TRAINER}__nnUNetPlans__{config}' / \
                 f'fold_{fold}' / 'checkpoint_final.pth'
    if checkpoint.exists():
        print(f'✓ Fold {fold} ({config}) ya completado en Drive, saltando')
        return True

    print(f'\n{"="*50}')
    print(f' {MODEL_FOLDER.upper()} — {config} — fold {fold}')
    print(f'{"="*50}')
    cmd = [
        'nnUNetv2_train',
        str(DATASET_ID), config, str(fold),
        '-tr', TRAINER,
        '--npz',
        '--c',
    ]
    result = subprocess.run(cmd, text=True)
    if result.returncode == 0:
        print(f'✓ Fold {fold} completado')
        backup_fold(config, fold)
        return True
    else:
        print(f'✗ Error en fold {fold}')
        return False

print('✓ Funciones listas')

✓ Funciones listas


# Entrenar 2D

In [ ]:
CONFIG = '2d'
restore_config(CONFIG)

print(f'Iniciando MedNeXt {CONFIG} — 500 epochs + early stopping\n')

for fold in range(5):
    ok = train_fold(CONFIG, fold)
    if not ok:
        print(f'\n⚠ Fallo en fold {fold}. Re-ejecuta esta celda.')
        break

print(f'\n✓ MedNeXt {CONFIG} completado')

  No hay checkpoints en Drive para 2d
Iniciando MedNeXt 2d — 500 epochs + early stopping


 MEDNEXT — 2d — fold 0
✗ Error en fold 0

⚠ Fallo en fold 0. Re-ejecuta esta celda.

✓ MedNeXt 2d completado


# Entrenar 3D lowres

In [ ]:
CONFIG = '3d_lowres'
restore_config(CONFIG)

print(f'Iniciando MedNeXt {CONFIG} — 500 epochs + early stopping\n')

for fold in range(5):
    ok = train_fold(CONFIG, fold)
    if not ok:
        print(f'\n⚠ Fallo en fold {fold}. Re-ejecuta esta celda.')
        break

print(f'\n✓ MedNeXt {CONFIG} completado')

# Entrenar 3D fullres

In [ ]:
CONFIG = '3d_fullres'
restore_config(CONFIG)

print(f'Iniciando MedNeXt {CONFIG} — 500 epochs + early stopping\n')

for fold in range(5):
    ok = train_fold(CONFIG, fold)
    if not ok:
        print(f'\n⚠ Fallo en fold {fold}. Re-ejecuta esta celda.')
        break

print(f'\n✓ MedNeXt {CONFIG} completado')

#Inferencia 3 configuraciones

In [ ]:
import os

input_dir = f'{NNUNET_RAW}/{DATASET_NAME}/imagesTs'

for config in ['2d', '3d_lowres', '3d_fullres']:
    output_dir = f'{NNUNET_RESULTS}/{DATASET_NAME}/inference_{config}'
    os.makedirs(output_dir, exist_ok=True)

    print(f'\n=== Inferencia MedNeXt {config} ===')
    cmd = [
        'nnUNetv2_predict',
        '-i', input_dir,
        '-o', output_dir,
        '-d', str(DATASET_ID),
        '-c', config,
        '-tr', TRAINER,
        '-f', '0', '1', '2', '3', '4',
        '--step_size', '0.5',
    ]
    result = subprocess.run(cmd, text=True)
    print(f'✓ {config}' if result.returncode == 0 else f'✗ Error: {result.returncode}')

    drive_inf = Path(DRIVE_RESULTS) / 'mednext' / f'inference_{config}'
    shutil.copytree(output_dir, str(drive_inf), dirs_exist_ok=True)
    print(f'✓ Predicciones {config} → Drive')

# Evaluación DSC + ID Rate + RMSD

In [ ]:
import pandas as pd

INVERSE_MAP = {v: k for k, v in LABEL_MAP.items()}

# --- DSC ---
def dsc_label(pred, gt, label):
    p, g  = (pred == label), (gt == label)
    inter = np.logical_and(p, g).sum()
    denom = p.sum() + g.sum()
    return float(2 * inter / denom) if denom > 0 else 1.0

def mean_dsc(pred, gt, labels):
    vals = [dsc_label(pred, gt, l) for l in labels if (gt==l).sum() > 0]
    return float(np.mean(vals)) if vals else 0.0

# --- ID Rate ---
def id_rate(pred, gt, labels, iou_thr=0.5):
    total, correct = 0, 0
    for l in labels:
        if (gt==l).sum() == 0:
            continue
        total += 1
        inter = np.logical_and(pred==l, gt==l).sum()
        union = np.logical_or(pred==l, gt==l).sum()
        if union > 0 and inter/union >= iou_thr:
            correct += 1
    return float(correct/total) if total > 0 else 0.0

# --- Centroides y RMSD ---
def get_centroids(seg, labels, affine):
    cents = {}
    for l in labels:
        idx = np.array(np.where(seg==l)).T
        if len(idx) == 0:
            continue
        c_vox  = idx.mean(axis=0)
        c_phys = (affine @ np.array([*c_vox, 1.0]))[:3]
        cents[l] = c_phys
    return cents

def compute_rmsd(pred, gt, affine, labels):
    pc = get_centroids(pred, labels, affine)
    gc = get_centroids(gt,   labels, affine)
    common = sorted(set(pc) & set(gc))
    if len(common) < 2:
        return float('nan')
    pred_phys = np.array([pc[l] for l in common])
    gt_phys   = np.array([gc[l] for l in common])
    try:
        from pycpd import AffineRegistration
        reg = AffineRegistration(X=gt_phys, Y=pred_phys)
        pred_aligned, _ = reg.register()
    except:
        pred_aligned = pred_phys - pred_phys.mean(0) + gt_phys.mean(0)
    return float(np.sqrt(np.mean(
        np.sum((pred_aligned - gt_phys)**2, axis=1)
    )))

# --- Evaluación completa ---
gt_dir  = Path(f'{NNUNET_RAW}/{DATASET_NAME}/labelsTs')
labels  = list(LABEL_MAP.values())
all_dfs = {}

for config in ['2d', '3d_lowres', '3d_fullres']:
    post_dir = Path(f'{NNUNET_RESULTS}/{DATASET_NAME}/inference_{config}_postprocessed')
    results  = []

    print(f'\nEvaluando {config}...')
    for pred_path in tqdm(sorted(post_dir.glob('*.nii.gz')), desc=config):
        gt_path = gt_dir / pred_path.name
        if not gt_path.exists():
            continue

        pred_nib = nib.load(str(pred_path))
        gt_nib   = nib.load(str(gt_path))
        pred_arr = np.array(pred_nib.dataobj, dtype=np.uint8)
        gt_arr   = np.array(gt_nib.dataobj,   dtype=np.uint8)

        results.append({
            'Case':      pred_path.stem.replace('.nii', ''),
            'DSC':       round(mean_dsc(pred_arr, gt_arr, labels),   4),
            'ID Rate':   round(id_rate(pred_arr, gt_arr, labels),    4),
            'RMSD (mm)': round(compute_rmsd(pred_arr, gt_arr,
                               gt_nib.affine, labels), 2),
        })

    df = pd.DataFrame(results)
    mean_row = {
        'Case':      'Mean',
        'DSC':       round(df['DSC'].mean(), 4),
        'ID Rate':   round(df['ID Rate'].mean(), 4),
        'RMSD (mm)': round(df['RMSD (mm)'].mean(), 2),
    }
    df = pd.concat([df, pd.DataFrame([mean_row])], ignore_index=True)
    all_dfs[config] = df

    print(f'\n--- {config.upper()} ---')
    print(df.to_string(index=False))

    # Alerta disociación DSC/ID Rate
    alert = df[(df['Case']!='Mean') & (df['DSC']>0.70) & (df['ID Rate']<0.30)]
    if not alert.empty:
        print(f'\n⚠ ALERTA — Fallo de etiquetado (DSC alta, ID Rate baja):')
        print(alert.to_string(index=False))

# Guardar en Drive
results_dir = Path(DRIVE_RESULTS) / 'nnunet' / 'evaluation'
results_dir.mkdir(parents=True, exist_ok=True)

for config, df in all_dfs.items():
    df.to_csv(str(results_dir / f'results_{config}.csv'), index=False)

print(f'\n✓ Resultados guardados en Drive: {results_dir}')

# Resumen comparativo final

In [ ]:
print('\n' + '='*60)
print(' RESUMEN nnU-Net — Comparación de configuraciones')
print('='*60)

summary = []
for config, df in all_dfs.items():
    mean_row = df[df['Case'] == 'Mean'].iloc[0]
    summary.append({
        'Config':    config,
        'DSC':       mean_row['DSC'],
        'ID Rate':   mean_row['ID Rate'],
        'RMSD (mm)': mean_row['RMSD (mm)'],
    })

df_summary = pd.DataFrame(summary)
print(df_summary.to_string(index=False))

mejor = df_summary.loc[df_summary['DSC'].idxmax(), 'Config']
print(f'\n✓ Mejor configuración por DSC: {mejor}')
print('='*60)

# Backup final completo a Drive

In [ ]:
import shutil

print('Backup final a Drive...')

for config in ['2d', '3d_lowres', '3d_fullres']:
    # Checkpoints
    local = Path(NNUNET_RESULTS) / DATASET_NAME / \
            f'{TRAINER}__nnUNetPlans__{config}'
    if local.exists():
        dest = Path(DRIVE_RESULTS) / 'nnunet' / DATASET_NAME / \
               f'{TRAINER}__nnUNetPlans__{config}'
        shutil.copytree(str(local), str(dest), dirs_exist_ok=True)
        print(f'  ✓ Checkpoints {config} → Drive')

    # Predicciones post-procesadas
    local_post = Path(NNUNET_RESULTS) / DATASET_NAME / \
                 f'inference_{config}_postprocessed'
    if local_post.exists():
        dest_post = Path(DRIVE_RESULTS) / 'nnunet' / \
                    f'inference_{config}_postprocessed'
        shutil.copytree(str(local_post), str(dest_post), dirs_exist_ok=True)
        print(f'  ✓ Predicciones {config} → Drive')

print('\n✓ Backup completo')
print(f'  Todo en: {DRIVE_RESULTS}/nnunet/')